In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(sys.prefix).parent
%cd {PROJECT_ROOT}

/home/younes/younes/Projects/Python/barid_internship


In [2]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["NIXTLA_ID_AS_COL"] = "true"
import numpy as np
np.set_printoptions(suppress=True)
np.random.seed(1)
import random
random.seed(1)
import pandas as pd
pd.set_option("max_colwidth", 100)
pd.set_option("display.precision", 3)
from utilsforecast.plotting import plot_series as plot_series_utils
import seaborn as sns
sns.set_style("whitegrid")
import matplotlib.pyplot as plt
plt.style.use("ggplot")
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "figure.constrained_layout.use": True,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.title_fontsize": 10,
    "grid.alpha": 1.0,
})
import matplotlib as mpl
from cycler import cycler
mpl.rcParams['axes.prop_cycle'] = cycler(color=["#000000", "#000000"])
from fpppy.utils import plot_series

mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#2f2fff"], name="black_and_blue"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55E00"], name="black_and_orange"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#000000"], name="black"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#0072B2", "#D55E00"],
        name='black_and_2color',
    ),
    force=True
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55E00", "#0072B2", "#009E73"],
        name='black_and_3color',
    ),
    force=True
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55E00", "#0072B2", "#009E73", "#CC79A7"],
        name='black_and_4color',
    ),
    force=True
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#D55E00", "#0072B2", "#009E73", "#CC79A7"],
        name='r_colors',
    ),
    force=True
)

In [11]:
import polars as pl
import altair as alt

# import seaborn as sns
# import matplotlib.pyplot as plt
# import numpy as np
# import pandas as pd
from functools import partial
from fpppy.utils import plot_series, plot_series_stacked, plot_diagnostics
from utilsforecast.plotting import plot_series as plot_series_utils
from utilsforecast.evaluation import evaluate
from statsforecast.utils import ConformalIntervals
from utilsforecast.losses import mase, rmsse, rmae, spis, nd
from coreforecast.scalers import boxcox, inv_boxcox, boxcox_lambda
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive
from mlforecast import MLForecast
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.linear_model import (
    LinearRegression,
    ElasticNet,
    ARDRegression,
    TweedieRegressor,
    Ridge,
    Lasso,
)
from sklearn.ensemble import HistGradientBoostingRegressor
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.utils import aggregate
from hierarchicalforecast.methods import BottomUp, TopDown, MinTrace
from utilsforecast.feature_engineering import trend, fourier, pipeline
from typing import Callable, Literal, Any, Mapping, Optional, Iterable, Sequence
from polars._typing import IntoExpr

# import calendar

# from scipy.stats import pearsonr
# from plotly import express as px
# from pathlib import Path
# from itertools import chain
from datetime import date, timedelta
from itertools import product

import importlib
import lib
import read_data

importlib.reload(lib)

cfg = pl.Config()
cfg.set_tbl_width_chars(10000)
cfg.set_fmt_str_lengths(100)
cfg.set_tbl_cols(-1)
cfg.set_tbl_rows(30)

alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [17]:
stat_cv_evaluation = pl.read_csv("results/stat_cv_evaluation.csv").drop("unique_id")
ml_cv_evaluation = pl.read_csv("results/ml_cv_evaluation.csv").drop(
    "unique_id",
    "SeasonalNaive",
)
neural_cv_evaluation = pl.read_csv("results/neural_cv_evaluation.csv").drop(
    "unique_id",
    "SeasonalNaive",
)

In [18]:
cv_evaluation = (
    stat_cv_evaluation.join(
        ml_cv_evaluation,
        on=["metric", "cutoff"],
    )
    .join(
        neural_cv_evaluation,
        on=["metric", "cutoff"],
        how="left",
    )
)
print(cv_evaluation)

shape: (40, 24)
┌─────────────────────────┬────────────────────┬───────────────┬─────────────┬─────────────┬─────────────┬─────────────┬────────────┬─────────────┬───────────────┬──────────────────┬─────────────┬───────────────┬──────────────────┬─────────────┬─────────────┬───────────────────────────────┬───────────────┬──────────────┬───────────────────┬─────────────┬────────────┬────────────┬─────────────────┐
│ cutoff                  ┆ metric             ┆ SeasonalNaive ┆ AutoETS     ┆ AutoARIMA   ┆ CES         ┆ AutoMFLES   ┆ AutoTBATS  ┆ AutoTheta   ┆ stat_ensemble ┆ LinearRegression ┆ ElasticNet  ┆ ARDRegression ┆ TweedieRegressor ┆ Lasso       ┆ Ridge       ┆ HistGradientBoostingRegressor ┆ LGBMRegressor ┆ XGBRegressor ┆ CatBoostRegressor ┆ ml_ensemble ┆ BiTCN      ┆ NHITS      ┆ neural_ensemble │
│ ---                     ┆ ---                ┆ ---           ┆ ---         ┆ ---         ┆ ---         ┆ ---         ┆ ---        ┆ ---         ┆ ---           ┆ ---              ┆

In [22]:
cfg.set_tbl_rows(300)

print(
    pl.DataFrame(cv_evaluation)
    .group_by("metric")
    .agg(pl.exclude("unique_id", "cutoff", "metric").mean())
    .unpivot(index="metric", variable_name="model", value_name="score")
    .sort("metric", "score")
    # .filter(
    #     pl.col("score").is_in(pl.col("score").unique().sort().head(10)).over("metric"),
    # )
    # .filter(metric="nd")
)

shape: (88, 3)
┌────────────────────┬───────────────────────────────┬─────────────┐
│ metric             ┆ model                         ┆ score       │
│ ---                ┆ ---                           ┆ ---         │
│ str                ┆ str                           ┆ f64         │
╞════════════════════╪═══════════════════════════════╪═════════════╡
│ mase               ┆ BiTCN                         ┆ 228.093866  │
│ mase               ┆ neural_ensemble               ┆ 237.952634  │
│ mase               ┆ NHITS                         ┆ 289.796279  │
│ mase               ┆ ml_ensemble                   ┆ 538.302449  │
│ mase               ┆ Ridge                         ┆ 576.801517  │
│ mase               ┆ CatBoostRegressor             ┆ 597.799729  │
│ mase               ┆ LinearRegression              ┆ 600.037066  │
│ mase               ┆ stat_ensemble                 ┆ 604.645129  │
│ mase               ┆ AutoARIMA                     ┆ 643.478845  │
│ mase             

In [20]:
chart = (
    alt.Chart(
        cv_evaluation.filter(metric="rmae_SeasonalNaive")
        .drop(
            "SeasonalNaive",
            "BiTCN",
            "NHITS",
            "neural_ensemble",
        )
        .with_columns(mean_rmae=pl.mean_horizontal(pl.selectors.numeric()))
        .to_pandas()
    )
    .mark_line(point=True)
    .encode(
        x=alt.X("cutoff:T", title="Cutoff"),
        y=alt.Y("mean_rmae:Q", title="RMAE"),
        # color=alt.Color("model:N", title="Model"),
        # tooltip=["cutoff:T", "model:N", "value:Q"],
    )
    .properties(
        width=800, height=400, title="Mean RMAE over Cross-validation windows across all models"
    )
)

chart

alt.Chart(...)

In [21]:
long_df = pl.DataFrame(cv_evaluation).unpivot(
    index=[
        # "unique_id",
        "cutoff",
        "metric",
    ],
    on=pl.selectors.exclude(
        [
            # "unique_id",
            "cutoff",
            "metric",
        ]
    ),
    variable_name="model",
    value_name="value",
)
# .filter(model)

chart = (
    alt.Chart(long_df.to_pandas())
    .mark_line(point=True)
    .encode(
        x=alt.X("cutoff:T", title="Cutoff"),
        y=alt.Y("value:Q", title="RMAE"),
        color=alt.Color("model:N", title="Model"),
        tooltip=["cutoff:T", "model:N", "value:Q"],
    )
    .properties(
        width=800, height=400, title="RMAE over Cross-validation windows by model"
    )
)

chart = lib.add_dropdown_filter(chart, long_df, "metric")

# chart.to_json(format="vega")
# chart.save(format="vega")
chart

alt.Chart(...)